### Process Orders data
1. Ingest the data into the data lakehouse- bronze_orders
2. Perform data quality checks and transform the data as required - silver_orders_clean
3. Explode the items array from the order object - silver_orders

## 1.Ingest the data into the data lakehouse- bronze_orders

![image_1781008922447.png](./image_1781008922447.png "image_1781008922447.png")

In [0]:
import dlt
from pyspark.sql.functions import *

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
@dlt.table(
    name="bronze_orders",
    table_properties={"quality":"bronze"},
    comment="Raw Orders data ingested from the source system"
)
def create_bronze_orders():
    return(
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes",True)
            .load("/Volumes/catalog_circuitbox/landing/operational_data/orders/")
    )

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:726)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:444)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:444)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

## 2. Perform data quality checks and transform the data as required - silver_orders_clean


![image_1781008899720.png](./image_1781008899720.png "image_1781008899720.png")

In [0]:
@dlt.table(
    name="silver_orders_clean",
    table_properties={'quality':'silver'},
    comment="Cleaned Order data"
)
@dlt.expect_or_fail("valid_customer_id","customer_id IS NOT NULL")
@dlt.expect_or_fail("valid_order_id","order_id IS NOT NULL")
@dlt.expect("valiad_order_status", "order_status IN ('Pending','Shipped','Cancelled','Completed') ")
@dlt.expect("valid_payment_method", "payment_method IN ('Creadit Card','Bank Transfer','PayPal') ")
def silver_orders_clean():
    return (
        spark.readStream.table("LIVE.bronze_orders")
            .select(
                col("order_id"),
                col("customer_id"),
                col("order_timestamp").cast("timestamp").alias("order_timestamp"),
                col("payment_method"),
                col("order_status"),
                col("items")
        )
    )         